In [ ]:
# ============================
# STEP 0: Download NIH ChestX-ray14 (224x224) from Kaggle 
# ============================

!pip install -q kaggle

import os, json
from google.colab import files

# 1) Upload your kaggle.json (from Kaggle > Account > API > Create New Token)
if not os.path.exists("kaggle.json"):
    print("👉 Upload kaggle.json from your computer")
    files.upload()  # choose kaggle.json

# 2) Move it into the right place
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 3) Download the resized ChestX-ray14 dataset
DATA_DIR = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)

!kaggle datasets download -d khanfashee/nih-chest-x-ray-14-224x224-resized -p {DATA_DIR}

# 4) Unzip
!unzip -q "{DATA_DIR}/nih-chest-x-ray-14-224x224-resized.zip" -d {DATA_DIR}

# 5) Inspect to see the exact folder names
!ls -R {DATA_DIR}


In [ ]:
# ============================
# STEP 1: Imports, device, constant paths, labels
# ============================

import os
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Paths (match downloaded dataset)
BASE_DIR = "/content/data"
CSV_PATH = os.path.join(BASE_DIR, "Data_Entry_2017.csv")
IMAGE_DIR = os.path.join(BASE_DIR, "images-224")
TRAIN_VAL_LIST = os.path.join(BASE_DIR, "train_val_list_NIH.txt")
TEST_LIST = os.path.join(BASE_DIR, "test_list_NIH.txt")

print("CSV_PATH:", CSV_PATH)
print("IMAGE_DIR:", IMAGE_DIR)

# 14 disease labels (order MUST match assignment)
DISEASE_LABELS = [
    "Atelectasis",
    "Cardiomegaly",
    "Effusion",
    "Infiltration",
    "Mass",
    "Nodule",
    "Pneumonia",
    "Pneumothorax",
    "Consolidation",
    "Edema",
    "Emphysema",
    "Fibrosis",
    "Pleural_Thickening",
    "Hernia",
]

IMG_SIZE = 224
BATCH_SIZE = 64  # reduce to 32 if OOM
NUM_WORKERS = 2
EPOCHS = 10       # adjust if you want longer training
MAX_SAMPLES = 15000  # subset for Colab speed

# Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)


Device: cuda
CSV_PATH: /content/data/Data_Entry_2017.csv
IMAGE_DIR: /content/data/images-224


In [ ]:
# ============================
# STEP 2: Build dataframe with multi-label targets
# ============================

# 1) Map image base names to full paths
img_map = {
    os.path.splitext(f)[0]: os.path.join(root, f)
    for root, _, files in os.walk(IMAGE_DIR)
    for f in files if f.lower().endswith((".png", ".jpg", ".jpeg"))
}
print("Total image files found on disk:", len(img_map))

# 2) Load CSV & keep only Image Index + labels
df = pd.read_csv(CSV_PATH)[["Image Index", "Finding Labels"]]
print("Total rows in CSV:", len(df))

# 3) Restrict to train/val subset using train_val_list_NIH.txt
if os.path.exists(TRAIN_VAL_LIST):
    with open(TRAIN_VAL_LIST, "r") as f:
        train_val_names = set(line.strip() for line in f if line.strip())
    df = df[df["Image Index"].isin(train_val_names)].reset_index(drop=True)
    print("Rows after filtering to train_val_list:", len(df))

# 4) Attach image paths
df["base_name"] = df["Image Index"].apply(lambda x: os.path.splitext(x)[0])
df["img_path"] = df["base_name"].map(img_map)
df = df[~df["img_path"].isna()].reset_index(drop=True)
print("Rows after matching with disk images:", len(df))

# 5) Sub-sample for speed (optional)
if len(df) > MAX_SAMPLES:
    df = df.sample(MAX_SAMPLES, random_state=42).reset_index(drop=True)
print("Rows after sub-sampling:", len(df))

# 6) Multi-label one-hot targets
for label in DISEASE_LABELS:
    df[label] = 0

def parse_labels(labels_str):
    return [] if labels_str == "No Finding" else [l.strip() for l in labels_str.split("|")]

for i, row in df.iterrows():
    labels = parse_labels(row["Finding Labels"])
    for lab in labels:
        if lab in DISEASE_LABELS:
            df.at[i, lab] = 1

# 7) Train / Val split (stratified by "has any disease")
train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    random_state=42,
    stratify=(df[DISEASE_LABELS].sum(axis=1) > 0)
)
print("Train size:", len(train_df))
print("Val size:", len(val_df))


In [ ]:
# ============================
# STEP 3: Class imbalance handling + Dataset + DataLoaders
# ============================

# Class imbalance: pos_weight for BCEWithLogitsLoss
label_counts = train_df[DISEASE_LABELS].sum(axis=0)
total = len(train_df)
pos_weight = (total - label_counts) / (label_counts + 1e-6)
pos_weight_tensor = torch.tensor(pos_weight.values, dtype=torch.float32).to(device)
print("pos_weight:", pos_weight_tensor)

# Dataset class
class ChestXrayDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.labels = DISEASE_LABELS

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row["img_path"]
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        labels = torch.tensor(row[self.labels].values.astype(np.float32))
        return img, labels

# Transforms
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = ChestXrayDataset(train_df, transform=train_transform)
val_dataset  = ChestXrayDataset(val_df,  transform=val_test_transform)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

len(train_loader), len(val_loader)


pos_weight: tensor([   8.3264,   59.0000,    8.7297,    5.1644,   23.6575,   17.7500,
          89.0000,   32.9623,   23.0000,   74.0000,   51.9412,   57.0645,
          39.9091, 1798.9982], device='cuda:0')


(29, 4)

In [ ]:
# ============================
# STEP 4: Model, loss, optimizer, evaluation helper
# ============================

# ResNet18 with 14-output head
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(DISEASE_LABELS))
model = model.to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=1
)

def evaluate(model, data_loader, threshold=0.5, return_probs=False):
    model.eval()
    total_loss = 0.0
    all_probs = []
    all_targets = []

    with torch.no_grad():
        for imgs, labels in data_loader:
            imgs = imgs.to(device)
            labels = labels.to(device)

            logits = model(imgs)
            loss = criterion(logits, labels)
            total_loss += loss.item() * imgs.size(0)

            probs = torch.sigmoid(logits).cpu().numpy()
            all_probs.append(probs)
            all_targets.append(labels.cpu().numpy())

    all_probs = np.vstack(all_probs)      # shape: [N, 14]
    all_targets = np.vstack(all_targets)  # shape: [N, 14]

    preds = (all_probs >= threshold).astype(int)
    macro_f1 = f1_score(all_targets, preds, average="macro", zero_division=0)
    mean_loss = total_loss / len(data_loader.dataset)

    if return_probs:
        return mean_loss, macro_f1, all_probs, all_targets
    else:
        return mean_loss, macro_f1


In [ ]:
# ============================
# STEP 5: Training loop (base model, threshold=0.5)
# ============================

best_val_f1 = 0.0
best_model_path = os.path.join(BASE_DIR, "model.pth")

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0

    for imgs, labels in train_loader:
        imgs = imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)

    train_loss = running_loss / len(train_loader.dataset)
    val_loss, val_f1 = evaluate(model, val_loader, threshold=0.5)
    scheduler.step(val_loss)

    print(f"Epoch [{epoch}/{EPOCHS}] "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Macro-F1: {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "label_order": DISEASE_LABELS,
            },
            best_model_path
        )
        print(f"✅ New best model (no thresholds yet) saved with Val Macro-F1 = {best_val_f1:.4f}")

print("Base training done. Best Val Macro-F1 (threshold=0.5):", best_val_f1)


Epoch [1/10] Train Loss: 1.3649 | Val Loss: 1.3480 | Val Macro-F1: 0.1084
✅ New best model (no thresholds yet) saved with Val Macro-F1 = 0.1084
Epoch [2/10] Train Loss: 1.0918 | Val Loss: 1.3221 | Val Macro-F1: 0.1279
✅ New best model (no thresholds yet) saved with Val Macro-F1 = 0.1279
Epoch [3/10] Train Loss: 0.9663 | Val Loss: 1.3889 | Val Macro-F1: 0.1442
✅ New best model (no thresholds yet) saved with Val Macro-F1 = 0.1442
Epoch [4/10] Train Loss: 0.8558 | Val Loss: 1.5277 | Val Macro-F1: 0.1456
✅ New best model (no thresholds yet) saved with Val Macro-F1 = 0.1456
Epoch [5/10] Train Loss: 0.7539 | Val Loss: 1.5272 | Val Macro-F1: 0.1524
✅ New best model (no thresholds yet) saved with Val Macro-F1 = 0.1524
Epoch [6/10] Train Loss: 0.6917 | Val Loss: 1.5941 | Val Macro-F1: 0.1535
✅ New best model (no thresholds yet) saved with Val Macro-F1 = 0.1535
Epoch [7/10] Train Loss: 0.6402 | Val Loss: 1.5999 | Val Macro-F1: 0.1577
✅ New best model (no thresholds yet) saved with Val Macro-F1 =

In [ ]:
# ============================
# STEP 6: Threshold tuning on validation set + save final model
# ============================

print("🔧 Tuning per-class thresholds on validation set...")

# Reload best weights
ckpt = torch.load(best_model_path, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.to(device)

val_loss, base_f1, val_probs, val_targets = evaluate(
    model, val_loader, threshold=0.5, return_probs=True
)
print(f"Validation Macro-F1 with global threshold 0.5: {base_f1:.4f}")

threshold_grid = np.linspace(0.1, 0.9, 9)
best_thresholds = np.full(len(DISEASE_LABELS), 0.5, dtype=np.float32)

for c in range(len(DISEASE_LABELS)):
    best_f1_c = 0.0
    best_t = 0.5
    y_true_c = val_targets[:, c]
    for t in threshold_grid:
        y_pred_c = (val_probs[:, c] >= t).astype(int)
        f1_c = f1_score(y_true_c, y_pred_c, zero_division=0)
        if f1_c > best_f1_c:
            best_f1_c = f1_c
            best_t = t
    best_thresholds[c] = best_t

print("Best per-class thresholds:", best_thresholds)

# Evaluate once more using tuned thresholds
final_preds = (val_probs >= best_thresholds.reshape(1, -1)).astype(int)
final_macro_f1 = f1_score(val_targets, final_preds, average="macro", zero_division=0)
print("Macro-F1 with tuned thresholds:", final_macro_f1)

# Save final checkpoint (weights + thresholds)
final_ckpt = {
    "model_state_dict": model.state_dict(),
    "label_order": DISEASE_LABELS,
    "thresholds": best_thresholds.tolist(),
}
torch.save(final_ckpt, best_model_path)
print("✅ Final model with thresholds saved to:", best_model_path)


🔧 Tuning per-class thresholds on validation set...
Validation Macro-F1 with global threshold 0.5: 0.1790
Best per-class thresholds: [0.7 0.5 0.5 0.5 0.5 0.4 0.1 0.5 0.7 0.7 0.3 0.7 0.5 0.5]
Macro-F1 with tuned thresholds: 0.2392505965174605
✅ Final model with thresholds saved to: /content/data/model.pth


In [ ]:
# ============================
# STEP 7: Export model.pth for DeepThink submission
# (DeepThink will run test.py and create predictions.csv itself)
# ============================

import torch

ckpt = torch.load(best_model_path, map_location="cpu")

# Ensure required metadata exists
ckpt["label_order"] = DISEASE_LABELS
if "thresholds" not in ckpt:
    ckpt["thresholds"] = [0.5] * len(DISEASE_LABELS)

torch.save(ckpt, "model.pth")
print("✅ Saved model.pth (for DeepThink) in current folder")
print("Checkpoint keys:", list(ckpt.keys()))


✅ Saved model.pth (for DeepThink) in current folder
Checkpoint keys: ['model_state_dict', 'label_order', 'thresholds']


In [ ]:
# ============================
# STEP 8: Create DeepThink test.py + ZIP (model.pth + test.py only)
# ============================

test_py = """#!/usr/bin/env python3
\'\'\'
Chest X-ray Multi-label Classification
\'\'\'

import sys
import os
import csv
import numpy as np
from PIL import Image, ImageFile

import torch
import torch.nn as nn
from torchvision import models, transforms

ImageFile.LOAD_TRUNCATED_IMAGES = True

# EXACT required output label order
DISEASE_LABELS = [
    "Atelectasis",
    "Cardiomegaly",
    "Effusion",
    "Infiltration",
    "Mass",
    "Nodule",
    "Pneumonia",
    "Pneumothorax",
    "Consolidation",
    "Edema",
    "Emphysema",
    "Fibrosis",
    "Pleural_Thickening",
    "Hernia",
]

IMG_SIZE = 224
BATCH_SIZE = 64  # speed knob: 16/32/64 (CPU usually 32, GPU can be higher)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Globals populated by load_model()
_TFM = None
_THRESHOLDS = None
_LABEL_MAP = None  # maps DISEASE_LABELS -> checkpoint indices


def _get_script_dir():
    try:
        return os.path.dirname(os.path.abspath(__file__))
    except NameError:
        return os.getcwd()


def _strip_module_prefix(state_dict):
    if not state_dict:
        return state_dict
    if any(k.startswith("module.") for k in state_dict.keys()):
        return {k.replace("module.", "", 1): v for k, v in state_dict.items()}
    return state_dict


def _build_resnet18(num_classes):
    # Avoid any internet/download dependency.
    try:
        m = models.resnet18(weights=None)
    except TypeError:
        m = models.resnet18(pretrained=False)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m


def load_model():
    \"\"\"Load your trained multi-label model\"\"\"
    global _TFM, _THRESHOLDS, _LABEL_MAP

    script_dir = _get_script_dir()
    model_path = os.path.join(script_dir, "model.pth")
    if not os.path.exists(model_path):
        model_path = os.path.join(os.getcwd(), "model.pth")
    if not os.path.exists(model_path):
        raise FileNotFoundError("model.pth not found. Put model.pth next to test.py (same folder).")

    ckpt = torch.load(model_path, map_location=device)

    ckpt_labels = ckpt.get("label_order") or ckpt.get("labels") or DISEASE_LABELS
    num_classes = len(ckpt_labels)

    model = _build_resnet18(num_classes)

    state_dict = ckpt.get("model_state_dict", ckpt)
    state_dict = _strip_module_prefix(state_dict)
    model.load_state_dict(state_dict, strict=True)
    model.to(device).eval()

    thr = ckpt.get("thresholds", [0.5] * num_classes)
    if isinstance(thr, (float, int)):
        thr = [float(thr)] * num_classes
    thr = list(thr)
    if len(thr) < num_classes:
        thr = thr + [0.5] * (num_classes - len(thr))
    if len(thr) > num_classes:
        thr = thr[:num_classes]
    _THRESHOLDS = torch.tensor(thr, dtype=torch.float32, device=device)

    ckpt_index = {lab: i for i, lab in enumerate(ckpt_labels)}
    _LABEL_MAP = [ckpt_index.get(lab, None) for lab in DISEASE_LABELS]

    _TFM = transforms.Compose(
        [
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )

    return model


def preprocess_image(image_path):
    \"\"\"Preprocess chest X-ray image for model input\"\"\"
    img = Image.open(image_path).convert("RGB")
    x = _TFM(img)                  # [C,H,W]
    return x


def predict_diseases(model, preprocessed_image_batch):
    \"\"\"
    Predict disease presence for a batch
    Returns: numpy int array shape [B, num_ckpt_classes] in checkpoint order
    \"\"\"
    with torch.inference_mode():
        x = preprocessed_image_batch.to(device)    # [B,C,H,W]
        logits = model(x)                          # [B,num_classes]
        probs = torch.sigmoid(logits)              # [B,num_classes]
        preds = (probs >= _THRESHOLDS).int().cpu().numpy()
    return preds


def main(input_dir):
    model = load_model()

    # collect image files
    filenames = [
        fn for fn in sorted(os.listdir(input_dir))
        if fn.lower().endswith((".png", ".jpg", ".jpeg"))
    ]

    fieldnames = ["filename"] + DISEASE_LABELS
    with open("predictions.csv", "w", newline="") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()

        batch_imgs = []
        batch_fns = []

        def flush_batch():
            if not batch_imgs:
                return

            batch_tensor = torch.stack(batch_imgs, dim=0)  # [B,C,H,W]
            preds_ckpt = predict_diseases(model, batch_tensor)  # [B,num_classes]

            for i, fname in enumerate(batch_fns):
                row = {"filename": fname}
                for lab, idx in zip(DISEASE_LABELS, _LABEL_MAP):
                    row[lab] = int(preds_ckpt[i, idx]) if idx is not None else 0
                writer.writerow(row)

            batch_imgs.clear()
            batch_fns.clear()

        for fn in filenames:
            path = os.path.join(input_dir, fn)
            try:
                x = preprocess_image(path)
                batch_imgs.append(x)
                batch_fns.append(fn)
            except Exception:
                # unreadable image -> all zeros
                row = {"filename": fn}
                for lab in DISEASE_LABELS:
                    row[lab] = 0
                writer.writerow(row)

            if len(batch_imgs) >= BATCH_SIZE:
                flush_batch()

        flush_batch()


if __name__ == "__main__":
    input_directory = sys.argv[1]
    main(input_directory)

"""

with open("test.py", "w", encoding="utf-8") as f:
    f.write(test_py)

print("✅ Wrote test.py")


